In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [2]:
df=pd.read_csv("House_Rent_Dataset.csv")


In [3]:
df.shape

(4746, 12)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4746 entries, 0 to 4745
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Posted On          4746 non-null   str  
 1   BHK                4746 non-null   int64
 2   Rent               4746 non-null   int64
 3   Size               4746 non-null   int64
 4   Floor              4746 non-null   str  
 5   Area Type          4746 non-null   str  
 6   Area Locality      4746 non-null   str  
 7   City               4746 non-null   str  
 8   Furnishing Status  4746 non-null   str  
 9   Tenant Preferred   4746 non-null   str  
 10  Bathroom           4746 non-null   int64
 11  Point of Contact   4746 non-null   str  
dtypes: int64(4), str(8)
memory usage: 445.1 KB


In [5]:
df["Posted On"]

0       2022-05-18
1       2022-05-13
2       2022-05-16
3       2022-07-04
4       2022-05-09
           ...    
4741    2022-05-18
4742    2022-05-15
4743    2022-07-10
4744    2022-07-06
4745    2022-05-04
Name: Posted On, Length: 4746, dtype: str

In [6]:
df["Floor"]

0       Ground out of 2
1            1 out of 3
2            1 out of 3
3            1 out of 2
4            1 out of 2
             ...       
4741         3 out of 5
4742         1 out of 4
4743         3 out of 5
4744       23 out of 34
4745         4 out of 5
Name: Floor, Length: 4746, dtype: str

In [7]:
df["Rent"]

0       10000
1       20000
2       17000
3       10000
4        7500
        ...  
4741    15000
4742    29000
4743    35000
4744    45000
4745    15000
Name: Rent, Length: 4746, dtype: int64

In [8]:
df["Floor"] = df["Floor"].astype(str).str.split(" out of").str[0]
df["Floor"] = df["Floor"].replace({"Ground": 0, "Upper Basement": -1, "Lower Basement": -2})
df["Floor"] = pd.to_numeric(df["Floor"], errors="coerce").fillna(0)

In [9]:
df["Floor"]

0        0
1        1
2        1
3        1
4        1
        ..
4741     3
4742     1
4743     3
4744    23
4745     4
Name: Floor, Length: 4746, dtype: int64

In [10]:
df.describe()

,BHK,Rent,Size,Floor,Bathroom
count,4746.000000,4.746000e+03,4746.000000,4746.000000,4746.000000
mean,2.083860,3.499345e+04,967.490729,3.436157,1.965866
std,0.832256,7.810641e+04,634.202328,5.773950,0.884532
min,1.000000,1.200000e+03,10.000000,-2.000000,1.000000
25%,2.000000,1.000000e+04,550.000000,1.000000,1.000000
50%,2.000000,1.600000e+04,850.000000,2.000000,2.000000
75%,3.000000,3.300000e+04,1200.000000,3.000000,2.000000
max,6.000000,3.500000e+06,8000.000000,76.000000,10.000000


In [11]:
df["Floor"] = df["Floor"].astype(str).str.split(" out of").str[0]
df["Floor"] = df["Floor"].replace({"Ground": 0, "Upper Basement": -1, "Lower Basement": -2})
df["Floor"] = pd.to_numeric(df["Floor"], errors="coerce").fillna(0)

In [12]:
df = df[df["Rent"] < df["Rent"].quantile(0.99)]

In [13]:
features = ["BHK", "Size", "Floor", "Bathroom", "City",
            "Area Type", "Furnishing Status", "Tenant Preferred"]

In [14]:
X = df[features]   
y = df["Rent"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((3754, 8), (939, 8))

In [16]:
cat_cols = ["City", "Area Type", "Furnishing Status", "Tenant Preferred"]


In [ ]:
pre = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough"   
)


In [18]:
model = Pipeline([
    ("pre", pre),
    ("rf", RandomForestRegressor(n_estimators=200, random_state=42))
])

In [19]:
model.fit(X_train, y_train)
'Training done'

'Training done'

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

pred = model.predict(X_test)   


"MAE:", mean_absolute_error(y_test, pred)


('MAE:', 9274.544205900449)

In [22]:

"R2 :", r2_score(y_test, pred)

('R2 :', 0.7625025603416504)

In [ ]:
import joblib


joblib.dump(model, "model.pkl")


sample = pd.DataFrame([{
    "BHK": 2, "Size": 900, "Floor": 2, "Bathroom": 2,
       "City": "Mumbai", "Area Type": "Super Area",
     "Furnishing Status": "Semi-Furnished",
    " Tenant Preferred": "Bachelors/Family"
}])

loaded = joblib.load("model.pkl")
print(loaded.predict(sample)[0])

54330.0
